<a href="https://colab.research.google.com/github/corrielynnyuill-debug/AIProject_Stocks-CLY/blob/main/AIProject_Stocks_Part1_CLY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Mount drive in Colab
from google.colab import drive
drive.mount('/content/drive')

# Load the datasets
import pandas as pd

# Set file path
DATA_PATH = "/content/drive/MyDrive/AIProject/"

# load datasets to DataFrame
prices = pd.read_csv(DATA_PATH + 'historical_stock_prices.csv')
stocks = pd.read_csv(DATA_PATH + 'historical_stocks.csv')

# Basic EDA dataset structures
print('Historical Stock Prices Dataset\n')
print(prices.head())
print('-'*80)
print(prices.info())
print('-'*80)
print(prices.isnull().sum())
print('-'*80)
print(prices.describe())
print('-'*80)
print('Historical Stock Prices shape before manipulation')
print(prices.shape)
print('-'*80)
print('\nHistorical Stocks Dataset\n')
print(stocks.head())
print('-'*80)
print(stocks.info())
print('-'*80)
print(stocks.describe())
print('Historical Stocks nefore manipulation')
print(stocks.shape)
print('-'*80)

# Handle missing values
# Fill missing sector/industry per ticker using mode
stocks['sector'] = stocks.groupby('ticker')['sector'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else '_missing_'))
stocks['industry'] = stocks.groupby('ticker')['industry'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else '_missing_'))

# Fallback to exchange level mode
stocks['sector'] = stocks.groupby('exchange')['sector'].transform(
    lambda x: x.replace('_missing_',x.mode()[0] if not x.mode().empty else '_missing_'))
stocks['industry'] = stocks.groupby('exchange')['industry'].transform(
    lambda x: x.replace('_missing',x.mode()[0] if not x.mode().empty else '_missing_'))

# Final fallback
stocks['sector'] = stocks['sector'].replace('_missing_','Unknown')
stocks['industry'] = stocks['industry'].replace('_missing_','Unkown')

# Covert date column to datetime
prices['date'] = pd.to_datetime(prices['date'], errors='coerce')

# Check for conversion errors
print(prices['date'].isna().sum())

# Index date for time series analysis
prices = prices.set_index('date').sort_index()

# Count full row duplicates in Historical Stock Prices
print(prices.duplicated().sum())

# Check duplicates by ticker and date
print(prices.duplicated(subset=['ticker', 'date']).sum)

# Drop duplicates safely keeping first entry
prices = prices[~prices.duplicated(subset=['ticker', 'date'],keep='first')]

# Count full row duplicates in Historical Stocks
print(stocks.duplicated().sum())

# Check duplicates by ticker
print(stocks.duplicated(subset=['ticker']).sum)

# Drop duplicates keeping first entry
stocks = stocks[~stocks.duplicated(subset=['ticker'],keep='first')]



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Historical Stock Prices Dataset

  ticker   open  close  adj_close    low   high   volume        date
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900  2013-05-08
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800  2013-05-09
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100  2013-05-10
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400  2013-05-13
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100  2013-05-14
--------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20973889 entries, 0 to 20973888
Data columns (total 8 columns):
 #   Column     Dtype  
---  ------     -----  
 0   ticker     object 
 1   open       float64
 2   close      float64
 3   adj_close  float64
 4   low        float64
 5   high       float64
 6   volume     int64  
 7   date    

KeyError: Index(['date'], dtype='object')